# Data Preprocessing - Combined Cycle Power Plant

Tahap preprocessing bertujuan menyiapkan dataset hasil cleaning agar dapat
digunakan oleh model Machine Learning.

Tahapan yang dilakukan:
1. Memisahkan fitur dan target.
2. Membagi data menjadi training dan testing untuk supervised learning.
3. Melakukan standardisasi fitur.
4. Menyiapkan data untuk clustering.
5. Menyimpan preprocessing object untuk digunakan kembali pada proses
   training dan prediksi.

In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [5]:
data_path = "../data/processed/CCPP_clean.csv"

df = pd.read_csv(data_path)

print("Shape:", df.shape)
df.head()

Shape: (9527, 5)


,AT,V,AP,RH,PE
0,14.96,41.76,1024.07,73.17,463.26
1,25.18,62.96,1020.04,59.08,444.37
2,5.11,39.40,1012.16,92.14,488.56
3,20.86,57.32,1010.24,76.64,446.48
4,10.82,37.50,1009.23,96.62,473.90


In [6]:
features = ["AT", "V", "AP", "RH"]
target = "PE"

In [7]:
X = df[features].copy()
y = df[target].copy()

print("X:", X.shape)
print("y:", y.shape)

X: (9527, 4)
y: (9527,)


In [8]:
print("Missing values:")
print(X.isnull().sum())

print("\nData type:")
print(X.dtypes)

Missing values:
AT    0
V     0
AP    0
RH    0
dtype: int64

Data type:
AT    float64
V     float64
AP    float64
RH    float64
dtype: object


In [9]:
print("\nTarget missing:", y.isnull().sum())


Target missing: 0


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [11]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (7621, 4)
X_test : (1906, 4)
y_train: (7621,)
y_test : (1906,)


In [12]:
regression_scaler = StandardScaler()

In [13]:
X_train_scaled = regression_scaler.fit_transform(X_train)

In [14]:
X_test_scaled = regression_scaler.transform(X_test)

In [15]:
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=features,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=features,
    index=X_test.index
)

In [16]:
X_train_scaled_df.describe().T

,count,mean,std,min,25%,50%,75%,max
AT,7621.0,1.901991e-16,1.000066,-2.405566,-0.816863,0.099437,0.805528,2.351111
V,7621.0,-1.468449e-16,1.000066,-2.279849,-0.988664,-0.173594,0.963879,2.150224
AP,7621.0,-6.372135e-15,1.000066,-3.430177,-0.694515,-0.051921,0.668256,3.385366
RH,7621.0,9.519277e-16,1.000066,-3.209120,-0.678436,0.114794,0.785726,1.837217


In [17]:
X_test_scaled_df.describe().T

,count,mean,std,min,25%,50%,75%,max
AT,1906.0,-0.002583,1.015567,-2.301809,-0.853919,0.068445,0.866165,2.093738
V,1906.0,0.004416,1.000106,-2.279849,-0.988664,-0.156252,0.963879,2.046961
AP,1906.0,0.007790,1.009580,-3.141769,-0.714332,-0.053608,0.676268,3.383680
RH,1906.0,0.022239,0.989905,-3.259679,-0.670066,0.131192,0.810835,1.836534


In [18]:
print("Train mean:")
print(X_train_scaled_df.mean())

print("\nTest mean:")
print(X_test_scaled_df.mean())

Train mean:
AT    1.901991e-16
V    -1.468449e-16
AP   -6.372135e-15
RH    9.519277e-16
dtype: float64

Test mean:
AT   -0.002583
V     0.004416
AP    0.007790
RH    0.022239
dtype: float64


In [23]:
X_cluster = df[features].copy()

In [25]:
cluster_scaler = StandardScaler()

X_cluster_scaled = cluster_scaler.fit_transform(
    X_cluster
)

In [26]:
X_cluster_scaled_df = pd.DataFrame(
    X_cluster_scaled,
    columns=features
)

X_cluster_scaled_df.describe().T

,count,mean,std,min,25%,50%,75%,max
AT,9527.0,-5.153616e-16,1.000052,-2.397664,-0.823243,0.092930,0.812973,2.344406
V,9527.0,-2.491039e-16,1.000052,-2.280800,-0.989577,-0.174482,0.963024,2.149405
AP,9527.0,8.929703e-15,1.000052,-3.425311,-0.698979,-0.053379,0.667133,3.377473
RH,9527.0,-8.882716e-16,1.000052,-3.270746,-0.681873,0.113992,0.788338,1.836483


In [28]:
joblib.dump(
    regression_scaler,
    "../models/regression_scaler.pkl"
)

['../models/regression_scaler.pkl']

In [29]:
joblib.dump(
    cluster_scaler,
    "../models/cluster_scaler.pkl"
)

['../models/cluster_scaler.pkl']

In [30]:
train_data = X_train.copy()
train_data[target] = y_train

test_data = X_test.copy()
test_data[target] = y_test

In [32]:
train_data.to_csv(
    "../data/processed/CCPP_train.csv",
    index=False
)

test_data.to_csv(
    "../data/processed/CCPP_test.csv",
    index=False
)

In [33]:
preprocessing_summary = {
    "dataset": "CCPP_clean.csv",
    "rows": len(df),
    "features": features,
    "target": target,
    "test_size": 0.20,
    "random_state": 42,
    "scaling": "StandardScaler",
    "regression_train_rows": len(X_train),
    "regression_test_rows": len(X_test),
    "clustering_features": features
}

preprocessing_summary

{'dataset': 'CCPP_clean.csv',
 'rows': 9527,
 'features': ['AT', 'V', 'AP', 'RH'],
 'target': 'PE',
 'test_size': 0.2,
 'random_state': 42,
 'scaling': 'StandardScaler',
 'regression_train_rows': 7621,
 'regression_test_rows': 1906,
 'clustering_features': ['AT', 'V', 'AP', 'RH']}

## Kesimpulan Preprocessing

Dataset hasil cleaning memiliki 9.527 observasi dan 5 variabel.

Empat variabel (`AT`, `V`, `AP`, dan `RH`) digunakan sebagai fitur,
sedangkan `PE` digunakan sebagai target pada supervised regression.

Untuk supervised learning, dataset dibagi menjadi data training sebesar
80% dan data testing sebesar 20% dengan `random_state=42`.

Standardisasi dilakukan menggunakan `StandardScaler`. Scaler dilatih
hanya menggunakan data training untuk mencegah data leakage, kemudian
digunakan untuk mentransformasi data training dan testing.

Untuk clustering, empat variabel input digunakan tanpa `PE`. Fitur
tersebut distandardisasi menggunakan scaler tersendiri.

Preprocessing object disimpan agar proses transformasi yang digunakan
saat training dapat diterapkan kembali secara konsisten ketika model
digunakan dalam aplikasi.